# CNN warden with curriculum training: SCRIPT_0.05 -> SCRIPT_0.02 -> SCRIPT_0.01

Trained from scratch at 0.01 bpnzac the CNN learned nothing (P_E 0.496, chance
0.491). A real warden would train where the signal is learnable and carry the
weights down. This notebook does that, one stage after another, with one paired
split per seed shared by every stage.

**Before you start:** *Accelerator -> GPU T4 x2*, *Internet -> On*, and
*+ Add Input ->* **both** datasets: the 5,000-image grid (`stego_5000.bin`) and
the low-payload one (`stego_script_low_5000.bin`). Then **Save Version -> Save &
Run All** and close the tab. Expected run time: about 4 hours.

The last cell prints the result per stage and writes `chain_results.zip`.

In [ ]:
# ---- Cell 1: preflight -------------------------------------------------
import json
import pathlib
import subprocess
import sys
import time
import zipfile

import torch

NOTEBOOK_T0 = time.time()
SESSION_LIMIT_MIN = 11.2 * 60      # Kaggle kills the session at 12 h
CHAIN = ['SCRIPT_0.05', 'SCRIPT_0.02', 'SCRIPT_0.01']
EPOCHS = ['40', '25', '25']
SEEDS = ['0', '1', '2', '3']
BATCH = "16"

print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
N_GPU = torch.cuda.device_count()
if N_GPU == 0:
    raise SystemExit("No GPU. Settings -> Accelerator -> GPU T4 x2, then rerun.")
for i in range(N_GPU):
    print("  GPU", i, torch.cuda.get_device_name(i))
try:
    import timm
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=True)
    import timm
print("timm", timm.__version__)

# Each dataset is ONE archive file; unpack it onto the session's local disk.
LOCAL = pathlib.Path("/kaggle/temp" if pathlib.Path("/kaggle/temp").is_dir() else "/tmp") / "stego_data"
print("looking for data under /kaggle/input ...", flush=True)
packed = [p for ext in ("*.bin", "*.zip")
          for p in sorted(pathlib.Path("/kaggle/input").rglob(ext))
          if zipfile.is_zipfile(p)]
for z in packed:
    t_z = time.time()
    dest = LOCAL / z.stem
    print("unpacking", z, "(%.2f GB) ->" % (z.stat().st_size / 1e9), dest, flush=True)
    with zipfile.ZipFile(z) as zf:
        zf.extractall(dest)
    print("  done in %.0f s" % (time.time() - t_z), flush=True)

# Find every cell of the chain, whichever dataset it is in.
WHERE = {}
roots = [pathlib.Path("/kaggle/input")] + ([LOCAL] if LOCAL.is_dir() else [])
for root in roots:
    for m in sorted(root.rglob("MANIFEST.json")):
        mf = json.loads(m.read_text())
        for c in mf["cells"]:
            if c["cell"] in CHAIN and c["cell"] not in WHERE:
                WHERE[c["cell"]] = (m.parent, m.parent / mf["cover_dir"], m.parent / c["stego_dir"])
missing = [c for c in CHAIN if c not in WHERE]
if missing:
    raise SystemExit("not found in any attached dataset: " + ", ".join(missing)
                     + " - attach BOTH datasets via '+ Add Input'.")
for c in CHAIN:
    print("  " + c.ljust(14), WHERE[c][2])

# The datasets must hold the SAME covers, or a pair would not be a pair.
DATA, COVER, _ = WHERE[CHAIN[-1]]
ref = {p.name: p.stat().st_size for p in COVER.glob("*.jpg")}
for c in CHAIN:
    other = {p.name: p.stat().st_size for p in WHERE[c][1].glob("*.jpg")}
    if other != ref:
        raise SystemExit("cover folders differ between datasets (" + c + ") - stop.")
print("covers:", COVER, "|", len(ref), "images, identical across datasets")

OUT = pathlib.Path("/kaggle/working/chain_results")
OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
%%writefile m4.py
"""M4 CNN steganalysis - reference implementation.

Paste into a Kaggle notebook (GPU on) or run as a script. Written to be correct
first and fast second; every choice that looks unusual is deliberate and is
explained in 02_PITFALLS.md.

Usage (Kaggle: attach the dataset first):

    python kaggle_notebook.py --data /kaggle/input/stego-eval \
        --out /kaggle/working/m4_results --null-test
    python kaggle_notebook.py --data /kaggle/input/stego-eval \
        --out /kaggle/working/m4_results --cells GMAS_0.1
    python kaggle_notebook.py --data /kaggle/input/stego-eval \
        --out /kaggle/working/m4_results --all

Requires: torch, timm, numpy, pillow.
"""
from __future__ import annotations

import argparse
import csv
import json
import math
import pathlib
import random
import shutil
import tempfile
import time

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset

SEEDS = [0, 1, 2, 3, 4]
# Early stopping on VALIDATION only: stop a seed after this many epochs with no
# strict improvement in validation P_E. None = always run every epoch.
PATIENCE = None


# ---------------------------------------------------------------- metrics ---
def p_error(sc: np.ndarray, ss: np.ndarray) -> float:
    """min over thresholds of 0.5*(P_FA + P_MD).

    Higher score must mean 'more likely stego'.

    WARNING - inverted scores do NOT show up as a value near 1.0. Because this
    is a min over thresholds, the degenerate threshold (call everything cover)
    always achieves 0.5, so backwards wiring collapses to ~0.5: it reads as
    'perfectly secure' and is indistinguishable from a passing null test. The
    only way to see it is to score the other direction as well - run_cell does,
    and reports p_e_reversed. If reversed is clearly LOWER, the scores are
    upside down.
    """
    best = 1.0
    for t in np.unique(np.concatenate([sc, ss])):
        best = min(best, 0.5 * ((sc >= t).mean() + (ss < t).mean()))
    return float(best)


def null_expectation(n_test_pairs: int) -> tuple[float, float]:
    """(mean, sd) of P_E when there is NO signal, at this test-set size.

    P_E = 0.5 * (1 - D+), where D+ is the one-sided two-sample
    Kolmogorov-Smirnov statistic. D+ is not zero under the null - it shrinks
    like 1/sqrt(n) - so a perfect pipeline reads BELOW 0.5: about 0.453 at 75
    test pairs, 0.466 at 150, 0.476 at 300. "Chance" is this number, not 0.50,
    and a P_E at or above it means "not detected at this sample size".

    Constants checked by simulation, 4000 trials per n.
    """
    n = max(int(n_test_pairs), 1)
    mean = 0.5 * (1.0 - math.sqrt(math.pi / 8.0) / math.sqrt(n / 2.0))
    return mean, 0.232 / math.sqrt(n)


def md_at_fa(sc: np.ndarray, ss: np.ndarray, target_fa: float = 0.01) -> float:
    """Missed detection at a fixed false-alarm rate - a realistic warden."""
    return float((ss < np.quantile(sc, 1 - target_fa)).mean())


# ---------------------------------------------------------------- dataset ---
class PairedSet(Dataset):
    """Cover/stego pairs for one cell.

    Loads JPEGs to float32 in [0,1] at native 512x512 and NEVER re-encodes.

    Augmentation is limited to flips and 90-degree rotations: these are the only
    transforms that permute pixels without resampling, so they preserve the
    embedding signal. Anything involving interpolation, cropping or
    re-compression destroys what we are trying to detect.
    """

    def __init__(self, cover_dir, stego_dir, stems, train: bool):
        self.cover_dir = pathlib.Path(cover_dir)
        self.stego_dir = pathlib.Path(stego_dir)
        self.stems = list(stems)
        self.train = train

    def __len__(self) -> int:
        return len(self.stems) * 2

    def __getitem__(self, i):
        stem = self.stems[i // 2]
        is_stego = i % 2
        d = self.stego_dir if is_stego else self.cover_dir
        x = np.asarray(Image.open(d / f"{stem}.jpg"), dtype=np.float32) / 255.0
        if self.train:
            if random.random() < 0.5:
                x = np.fliplr(x)
            if random.random() < 0.5:
                x = np.flipud(x)
            k = random.randint(0, 3)
            if k:
                x = np.rot90(x, k)
        x = np.ascontiguousarray(x)[None]  # (1, H, W)
        return torch.from_numpy(x), torch.tensor(float(is_stego)), stem


# ------------------------------------------------------------------ model ---
def build_model() -> nn.Module:
    """EfficientNet-B0, ImageNet weights, single-channel input.

    No resize: the network is fully convolutional and accepts 512x512 directly.
    Downsampling to 224 would average away the residual the detector needs.
    `in_chans=1` lets timm adapt the stem rather than us faking a 3-channel
    image. Inputs are scaled to [0,1] only - no ImageNet mean/std, which is
    meaningless on a replicated grayscale channel.
    """
    import timm

    return timm.create_model("efficientnet_b0", pretrained=True,
                             num_classes=1, in_chans=1)


# ---------------------------------------------------------------- scoring ---
def score_loader(model, dl, device, use_amp):
    """Score every image in a loader -> (cover scores, stego scores, rows)."""
    model.eval()
    sc, ss, keep = [], [], []
    with torch.no_grad():
        for x, y, stem in dl:
            with torch.cuda.amp.autocast(enabled=use_amp):
                o = model(x.to(device)).squeeze(1).float().cpu().numpy()
            for oi, yi, si in zip(o, y.numpy(), stem):
                (ss if yi > 0.5 else sc).append(oi)
                keep.append((si, "stego" if yi > 0.5 else "cover", float(oi)))
    return np.asarray(sc), np.asarray(ss), keep


# --------------------------------------------------------------- one cell ---
def run_cell(cover_dir, stego_dir, stems, device, epochs, bs, lr, out_scores):
    per_seed_pe, per_seed_md, best_epochs = [], [], []
    per_seed_rev, per_seed_val = [], []
    epochs_run = []
    rows = []
    n_tr = n_va = n_te = 0

    for seed in SEEDS:
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

        # PAIRED split: the index decides, so cover i and stego i stay together.
        # Splitting the two classes independently would put a cover in training
        # and its own stego in test - the network then recognises the photograph
        # instead of the embedding. See 02_PITFALLS.md section 1.
        #
        # Three-way: 40% train / 10% validation / 50% test. The validation
        # slice exists only to choose the epoch. Choosing it on the test set
        # instead - the common shortcut - takes a min over ~40 noisy estimates
        # and drags P_E down by ~0.07 with no signal present at all, which both
        # fails the null test on a correct pipeline and understates the
        # security of every genuinely hard-to-detect method.
        idx = np.random.default_rng(seed).permutation(len(stems))
        half = len(stems) // 2
        n_val = max(2, int(round(0.2 * half)))
        va = [stems[i] for i in idx[:n_val]]
        tr = [stems[i] for i in idx[n_val:half]]
        te = [stems[i] for i in idx[half:]]
        assert not (set(tr) & set(te)), "train/test stems overlap"
        assert not (set(va) & set(te)), "val/test stems overlap"
        assert not (set(tr) & set(va)), "train/val stems overlap"
        n_tr, n_va, n_te = len(tr), len(va), len(te)

        g = torch.Generator()
        g.manual_seed(seed)
        dl_tr = DataLoader(PairedSet(cover_dir, stego_dir, tr, True),
                           batch_size=bs, shuffle=True, num_workers=2,
                           drop_last=True, generator=g)
        dl_va = DataLoader(PairedSet(cover_dir, stego_dir, va, False),
                           batch_size=bs, shuffle=False, num_workers=2)
        dl_te = DataLoader(PairedSet(cover_dir, stego_dir, te, False),
                           batch_size=bs, shuffle=False, num_workers=2)

        model = build_model().to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=lr)
        lossf = nn.BCEWithLogitsLoss()
        use_amp = device.type == "cuda"
        scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

        best_val, chosen, best_ep = 2.0, None, -1
        best_strict, since_better, ran = 2.0, 0, 0
        for ep in range(epochs):
            ran = ep + 1
            model.train()
            for x, y, _ in dl_tr:
                x = x.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)
                opt.zero_grad(set_to_none=True)
                with torch.cuda.amp.autocast(enabled=use_amp):
                    loss = lossf(model(x).squeeze(1), y)
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()

            # Epoch selection looks at VALIDATION only. The test set is
            # scored at the chosen epoch and never influences the choice.
            # `<=` keeps the later epoch on ties: the validation slice is
            # small, ties are common, and a longer-trained model is the safer
            # tie-break.
            vc, vs, _ = score_loader(model, dl_va, device, use_amp)
            val_pe = p_error(vc, vs)
            if val_pe <= best_val:
                best_val = val_pe
                chosen = score_loader(model, dl_te, device, use_amp)
                best_ep = ep
            # Patience counts STRICT improvements: with `<=` alone, a cell at
            # chance ties forever and would never stop.
            if val_pe < best_strict:
                best_strict, since_better = val_pe, 0
            else:
                since_better += 1
            if PATIENCE is not None and since_better >= PATIENCE:
                break

        sc, ss, keep = chosen
        test_pe = p_error(sc, ss)
        rev_pe = p_error(ss, sc)
        per_seed_pe.append(test_pe)
        per_seed_rev.append(rev_pe)
        per_seed_val.append(best_val)
        per_seed_md.append(md_at_fa(sc, ss))
        best_epochs.append(best_ep)
        epochs_run.append(ran)
        rows += [(seed, s, lbl, v) for s, lbl, v in keep]
        print(f"    seed {seed}: P_E {test_pe:.4f} "
              f"(val {best_val:.4f}, epoch {best_ep} of {ran} run)", flush=True)

    # Inversion guard. For a detectable cell with correctly oriented scores,
    # the forward P_E is low and the reversed one sits at ~0.5. If it is the
    # other way round the scores are upside down - and the forward number
    # would be reporting ~0.5, i.e. "secure", for a method that is not.
    #
    # Both conditions are needed. At chance level the two directions are just
    # noise around the same value, so "reversed < forward" alone fires at
    # random - including on the null test, where it would be a false alarm on
    # the one check that must be trusted. Requiring the reversed direction to
    # sit significantly BELOW chance means it fires only on a real signal
    # pointing the wrong way.
    chance, chance_sd = null_expectation(n_te)
    rev_mean, fwd_mean = float(np.mean(per_seed_rev)), float(np.mean(per_seed_pe))
    if rev_mean < chance - 3.0 * chance_sd and rev_mean < fwd_mean - 0.05:
        print("  !! SCORES LOOK INVERTED: reversed P_E "
              f"{np.mean(per_seed_rev):.4f} is lower than forward "
              f"{np.mean(per_seed_pe):.4f}. Higher score must mean 'stego'.",
              flush=True)

    out_scores.parent.mkdir(parents=True, exist_ok=True)
    with out_scores.open("w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["seed", "image", "label", "score"])
        w.writerows(rows)

    return dict(
        p_e_mean=float(np.mean(per_seed_pe)),
        p_e_std=float(np.std(per_seed_pe)),
        p_e_per_seed=[float(v) for v in per_seed_pe],
        md_at_fa1_mean=float(np.mean(per_seed_md)),
        md_at_fa1_std=float(np.std(per_seed_md)),
        best_epoch_per_seed=best_epochs,
        epochs_run_per_seed=epochs_run,
        patience=PATIENCE,
        p_e_reversed_mean=float(np.mean(per_seed_rev)),
        val_p_e_per_seed=[float(v) for v in per_seed_val],
        # What P_E reads at this test size with NO signal. Not 0.50.
        p_e_chance_level=float(null_expectation(n_te)[0]),
        epoch_selection="validation",
        n_train_pairs=n_tr, n_val_pairs=n_va, n_test_pairs=n_te,
    )


def null_test(cover_dir, out, device, args) -> bool:
    """Covers labelled against other covers. Must return chance-level P_E.

    "Chance" is NOT 0.50 - see null_expectation. The gate is one-sided: a leak
    can only push P_E down, so we only test the lower bound.

    This is the check that catches label leakage, ordering artifacts and
    train/test contamination. Nothing downstream is trustworthy until it passes.
    """
    print("\nNULL TEST (cover vs cover) - expect chance-level P_E")
    stems = sorted((p.stem for p in cover_dir.glob("*.jpg")),
                   key=lambda s: (len(s), s))
    # The two groups MUST be drawn at random. BOSSbase is ordered by source
    # (camera), so "first half vs second half by file number" compares two
    # different kinds of image, and any competent detector separates them:
    # at 2,000 covers that read P_E 0.29 here and 0.31 under DCTR, with no leak
    # anywhere. Random groups of the same covers give 0.48.
    stems = [stems[i] for i in
             np.random.default_rng(20261002).permutation(len(stems))]
    half = len(stems) // 2
    a, b = stems[:half], stems[half:2 * half]

    # Build a fake 'stego' folder whose file N.jpg is a *different cover*, named
    # to pair with cover N.jpg. There is no embedded signal to find.
    with tempfile.TemporaryDirectory() as td:
        fake = pathlib.Path(td)
        for sa, sb in zip(a, b):
            shutil.copyfile(cover_dir / f"{sb}.jpg", fake / f"{sa}.jpg")
        r = run_cell(cover_dir, fake, a, device, max(5, args.epochs // 4),
                     args.batch_size, args.lr, out / "scores" / "NULL_TEST.csv")

    mean, sd = null_expectation(r["n_test_pairs"])
    # Seeds reuse the same images under different splits, so they are not
    # independent draws; 2.5 sd of a SINGLE seed is a deliberately loose floor.
    # A real leak (the network recognising photographs) lands far below it.
    floor = mean - 2.5 * sd
    r["null_expected"], r["null_floor"] = mean, floor
    print(f"\n  NULL TEST P_E = {r['p_e_mean']:.4f}   "
          f"(chance at {r['n_test_pairs']} test pairs = {mean:.4f}, "
          f"pass if >= {floor:.4f})")
    ok = r["p_e_mean"] >= floor
    print("  " + ("PASS - pipeline finds no signal where none exists" if ok else
                  "FAIL - PIPELINE IS LEAKING. Stop and debug (02_PITFALLS.md 2)"))
    (out / "null_test.json").write_text(json.dumps(r, indent=2), encoding="utf-8")
    return ok


def resolve_data_root(given: pathlib.Path) -> pathlib.Path:
    """Find the folder that actually contains MANIFEST.json.

    Kaggle mounts a dataset at /kaggle/input/<slug>/, and because our zip has a
    top-level `dataset/` folder the real root is usually
    /kaggle/input/<slug>/dataset. Rather than making anyone get that right, we
    search: the given path, then one or two levels down.
    """
    if (given / "MANIFEST.json").exists():
        return given
    for cand in sorted(given.glob("*/MANIFEST.json")) + sorted(given.glob("*/*/MANIFEST.json")):
        print(f"note: using data root {cand.parent} (MANIFEST.json found there)")
        return cand.parent
    raise SystemExit(
        "No MANIFEST.json under " + str(given) + "\n"
        "Point --data at the folder containing MANIFEST.json. On Kaggle that is\n"
        "usually /kaggle/input/<dataset-slug>/dataset\n"
        "Tip: run  !find /kaggle/input -name MANIFEST.json  to locate it.")


# ------------------------------------------------------------------- main ---
def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True, help="dataset root (has MANIFEST.json)")
    ap.add_argument("--out", required=True, help="output dir for m4_results")
    ap.add_argument("--cells", nargs="*", default=None, help="specific cell names")
    ap.add_argument("--all", action="store_true", help="run every cell")
    ap.add_argument("--null-test", action="store_true", help="run the null test first")
    ap.add_argument("--epochs", type=int, default=40)
    ap.add_argument("--time-budget-min", type=float, default=None,
                    help="stop cleanly before starting a cell that would not "
                         "finish inside this many minutes from now")
    ap.add_argument("--patience", type=int, default=None,
                    help="early-stop a seed after N epochs without a strict "
                         "validation improvement")
    ap.add_argument("--seeds", type=int, default=5,
                    help="number of seeds (0..N-1); 5 for the 300-image grid")
    ap.add_argument("--batch-size", type=int, default=8)
    ap.add_argument("--lr", type=float, default=1e-4)
    args = ap.parse_args()

    global SEEDS, PATIENCE
    SEEDS = list(range(args.seeds))
    PATIENCE = args.patience

    data = resolve_data_root(pathlib.Path(args.data))
    out = pathlib.Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"device: {device}")
    if device.type != "cuda":
        print("WARNING: no GPU. This will take days on CPU.")

    manifest = json.loads((data / "MANIFEST.json").read_text(encoding="utf-8"))
    cover_dir = data / manifest["cover_dir"]

    if args.null_test:
        if not null_test(cover_dir, out, device, args):
            return 1
        if not (args.all or args.cells):
            return 0

    cells = manifest["cells"]
    if args.cells:
        cells = [c for c in cells if c["cell"] in args.cells]
        missing = set(args.cells) - {c["cell"] for c in cells}
        if missing:
            print(f"unknown cell names: {sorted(missing)}")
            return 1
    elif not args.all:
        print("nothing to do: pass --all or --cells NAME")
        return 0

    res_path = out / "results.json"
    results = (json.loads(res_path.read_text(encoding="utf-8"))
               if res_path.exists() else [])
    done = {(r["cell"], r["detector"]) for r in results}

    t_begin, longest = time.time(), 0.0
    for c in cells:
        if (c["cell"], "EfficientNet-B0") in done:
            print(f"skip {c['cell']} (already in results.json)")
            continue
        # A Kaggle session that hits its 12-hour cap is killed and its output
        # is lost. Stopping one cell early keeps everything finished so far;
        # the remaining cells are picked up by a rerun.
        if args.time_budget_min is not None:
            used = (time.time() - t_begin) / 60
            if used + longest * 1.15 > args.time_budget_min:
                print(f"TIME BUDGET: {used:.0f} min used, next cell needs about "
                      f"{longest * 1.15:.0f}; stopping cleanly before {c['cell']}")
                break
        stego_dir = data / c["stego_dir"]
        stems = sorted((p.stem for p in stego_dir.glob("*.jpg")),
                       key=lambda s: (len(s), s))
        # Exact pairing only. SCRIPT cells ship 299 stegos against 300 covers;
        # dropping the unmatched cover keeps the pairing exact. Never pad or
        # duplicate to make the counts line up.
        stems = [s for s in stems if (cover_dir / f"{s}.jpg").exists()]
        print(f"\n{c['cell']}  ({len(stems)} pairs)", flush=True)

        t0 = time.time()
        try:
            r = run_cell(cover_dir, stego_dir, stems, device, args.epochs,
                         args.batch_size, args.lr,
                         out / "scores" / f"{c['cell']}.csv")
            entry = dict(cell=c["cell"], method=c["method"],
                         payload_bpnzac=c["payload_bpnzac"], arm=c["arm"],
                         detector="EfficientNet-B0", status="ok",
                         seeds=SEEDS, epochs=args.epochs,
                         train_minutes=round((time.time() - t0) / 60, 1), **r)
            longest = max(longest, (time.time() - t0) / 60)
            print(f"  -> P_E {entry['p_e_mean']:.4f} +/- {entry['p_e_std']:.4f}"
                  f"   (chance = {entry['p_e_chance_level']:.4f})")
        except Exception as exc:  # noqa: BLE001
            # Record failures as failures. Never substitute a default value -
            # that is exactly the defect we found in four published drivers.
            entry = dict(cell=c["cell"], method=c["method"],
                         payload_bpnzac=c["payload_bpnzac"], arm=c["arm"],
                         detector="EfficientNet-B0", status="failed",
                         error=str(exc)[:300])
            print(f"  FAILED: {exc}")

        results.append(entry)
        # Checkpoint after every cell so a session timeout costs one cell, not all.
        res_path.write_text(json.dumps(results, indent=2), encoding="utf-8")

    print(f"\nwrote {res_path}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
%%writefile m4_chain.py
"""Curriculum ("chain") training of the CNN warden for very low payloads.

Trained from scratch at 0.01 bpnzac the network learns nothing: the signal is
too weak to find from an ImageNet starting point. The standard remedy in
steganalysis is to train where the signal is learnable and carry the weights
down: here SCRIPT 0.05 -> 0.02 -> 0.01. A warden would do this, so a security
figure measured against a from-scratch network is only an upper bound.

For each seed, ONE paired 40/10/50 split is drawn over the images present in
every stage and used for all of them. A test image is therefore never seen in
training at any payload - not as a cover, not as a stego.

Per stage:
  - the network as it arrives from the previous stage is scored first
    ("transfer": a warden trained at the higher payload, applied unchanged);
  - it is then fine-tuned, and the epoch is chosen on VALIDATION only, with the
    untouched arriving network as one of the candidates;
  - the chosen weights go on to the next stage.

Everything else (model, augmentation, optimiser, P_E) is imported from m4.py so
the two cannot drift apart.

    python m4_chain.py --cover DIR --stage SCRIPT_0.05=DIR --stage SCRIPT_0.02=DIR \
        --stage SCRIPT_0.01=DIR --epochs 40 25 25 --seeds 0 1 --out OUT
"""
from __future__ import annotations

import argparse
import copy
import csv
import json
import pathlib
import random
import time

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

import m4


def split(stems: list[str], seed: int):
    """Same rule as m4.run_cell: 10% validation, 40% train, 50% test, paired."""
    idx = np.random.default_rng(seed).permutation(len(stems))
    half = len(stems) // 2
    n_val = max(2, int(round(0.2 * half)))
    va = [stems[i] for i in idx[:n_val]]
    tr = [stems[i] for i in idx[n_val:half]]
    te = [stems[i] for i in idx[half:]]
    assert not (set(tr) & set(te)) and not (set(va) & set(te)) and not (set(tr) & set(va))
    return tr, va, te


def run_seed(seed, cover_dir, stages, epochs, stems, device, bs, lr, workers):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    tr, va, te = split(stems, seed)
    use_amp = device.type == "cuda"
    lossf = nn.BCEWithLogitsLoss()
    model = m4.build_model().to(device)
    out, score_rows = [], {}

    for si, (name, stego_dir) in enumerate(stages):
        g = torch.Generator()
        g.manual_seed(seed * 100 + si)
        dl_tr = DataLoader(m4.PairedSet(cover_dir, stego_dir, tr, True), batch_size=bs,
                           shuffle=True, num_workers=workers, drop_last=True, generator=g)
        dl_va = DataLoader(m4.PairedSet(cover_dir, stego_dir, va, False),
                           batch_size=bs, shuffle=False, num_workers=workers)
        dl_te = DataLoader(m4.PairedSet(cover_dir, stego_dir, te, False),
                           batch_size=bs, shuffle=False, num_workers=workers)
        # Fresh optimiser and scaler per stage: the momentum built up at the
        # previous payload is not information about this one.
        opt = torch.optim.AdamW(model.parameters(), lr=lr)
        scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

        best_val, chosen, best_ep, best_state = 2.0, None, None, None
        transfer = None
        if si > 0:
            vc, vs, _ = m4.score_loader(model, dl_va, device, use_amp)
            best_val = m4.p_error(vc, vs)
            chosen = m4.score_loader(model, dl_te, device, use_amp)
            best_ep = -1
            best_state = copy.deepcopy(model.state_dict())
            transfer = dict(test_p_e=m4.p_error(chosen[0], chosen[1]), val_p_e=best_val,
                            md_at_fa1=m4.md_at_fa(chosen[0], chosen[1]))
            score_rows[name + "__transfer"] = [(seed, s, lbl, v) for s, lbl, v in chosen[2]]
            print(f"    seed {seed} {name}: transfer from {stages[si - 1][0]} "
                  f"P_E {transfer['test_p_e']:.4f} (val {best_val:.4f})", flush=True)

        t0 = time.time()
        for ep in range(epochs[si]):
            model.train()
            for x, y, _ in dl_tr:
                x = x.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)
                opt.zero_grad(set_to_none=True)
                with torch.cuda.amp.autocast(enabled=use_amp):
                    loss = lossf(model(x).squeeze(1), y)
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
            vc, vs, _ = m4.score_loader(model, dl_va, device, use_amp)
            val_pe = m4.p_error(vc, vs)
            if val_pe <= best_val:          # validation only; later epoch on ties
                best_val = val_pe
                chosen = m4.score_loader(model, dl_te, device, use_amp)
                best_ep = ep
                best_state = copy.deepcopy(model.state_dict())

        model.load_state_dict(best_state)   # the chosen weights go to the next stage
        sc, ss, keep = chosen
        r = dict(seed=seed, stage=name, stage_index=si, epochs=epochs[si],
                 test_p_e=m4.p_error(sc, ss), test_p_e_reversed=m4.p_error(ss, sc),
                 val_p_e=best_val, md_at_fa1=m4.md_at_fa(sc, ss), best_epoch=best_ep,
                 transfer=transfer, minutes=round((time.time() - t0) / 60, 1),
                 n_train_pairs=len(tr), n_val_pairs=len(va), n_test_pairs=len(te))
        out.append(r)
        score_rows[name] = [(seed, s, lbl, v) for s, lbl, v in keep]
        print(f"    seed {seed} {name}: P_E {r['test_p_e']:.4f} (val {best_val:.4f}, "
              f"epoch {best_ep} of {epochs[si]}; -1 = arriving weights)  "
              f"{r['minutes']:.0f} min", flush=True)
    return out, score_rows


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--cover", required=True)
    ap.add_argument("--stage", action="append", required=True, metavar="NAME=DIR",
                    help="in training order, highest payload first")
    ap.add_argument("--epochs", type=int, nargs="+", required=True,
                    help="one value per stage, or one value for all")
    ap.add_argument("--seeds", type=int, nargs="+", default=[0])
    ap.add_argument("--out", required=True)
    ap.add_argument("--batch-size", type=int, default=16)
    ap.add_argument("--lr", type=float, default=1e-4)
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--limit", type=int, default=None, help="smoke test: first N pairs")
    ap.add_argument("--time-budget-min", type=float, default=None)
    args = ap.parse_args()

    cover_dir = pathlib.Path(args.cover)
    stages = [(s.split("=", 1)[0], pathlib.Path(s.split("=", 1)[1])) for s in args.stage]
    epochs = args.epochs * len(stages) if len(args.epochs) == 1 else args.epochs
    if len(epochs) != len(stages):
        raise SystemExit("--epochs needs one value, or one per stage")
    out = pathlib.Path(args.out)
    (out / "scores").mkdir(parents=True, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"device: {device}")

    # Only images present in EVERY stage, so one split serves the whole chain.
    common = {p.stem for p in cover_dir.glob("*.jpg")}
    for name, d in stages:
        have = {p.stem for p in d.glob("*.jpg")}
        print(f"  {name}: {len(have)} stego images")
        common &= have
    stems = sorted(common, key=lambda s: (len(s), s))
    if args.limit:
        stems = stems[:args.limit]
    chance, sd = m4.null_expectation(len(stems) - len(stems) // 2)
    print(f"chain {' -> '.join(n for n, _ in stages)}: {len(stems)} pairs in every "
          f"stage, epochs {epochs}, seeds {args.seeds}, chance {chance:.4f} +/- {sd:.4f}",
          flush=True)

    res_path = out / "seed_results.json"
    results = json.loads(res_path.read_text()) if res_path.exists() else []
    t_begin, longest = time.time(), 0.0
    for seed in args.seeds:
        if any(r["seed"] == seed for r in results):
            print(f"skip seed {seed} (already done)")
            continue
        if args.time_budget_min is not None:
            used = (time.time() - t_begin) / 60
            if used + longest * 1.15 > args.time_budget_min:
                print(f"TIME BUDGET: {used:.0f} min used; stopping before seed {seed}")
                break
        t0 = time.time()
        rows, score_rows = run_seed(seed, cover_dir, stages, epochs, stems, device,
                                    args.batch_size, args.lr, args.workers)
        longest = max(longest, (time.time() - t0) / 60)
        # Checkpoint after every seed: a session timeout costs one seed, not all.
        for name, rr in score_rows.items():
            f = out / "scores" / f"{name}.csv"
            new = not f.exists()
            with f.open("a", newline="", encoding="utf-8") as fh:
                w = csv.writer(fh)
                if new:
                    w.writerow(["seed", "image", "label", "score"])
                w.writerows(rr)
        results += rows
        res_path.write_text(json.dumps(results, indent=2), encoding="utf-8")
    print(f"wrote {res_path}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
# ---- Cell 2: null test (gate) ------------------------------------------
# Covers against other covers must read chance. If this fails the pipeline is
# leaking and the chain does not run.
p = subprocess.run([sys.executable, "m4.py", "--data", str(DATA), "--seeds", "3",
                    "--epochs", "40", "--batch-size", BATCH,
                    "--out", str(OUT / "null"), "--null-test"])
if p.returncode != 0:
    raise RuntimeError("CNN null test failed - do not run the chain; read the output above.")
print("elapsed: %.0f min" % ((time.time() - NOTEBOOK_T0) / 60))

In [ ]:
# ---- Cell 3: the chain, one process per GPU ----------------------------
# Seeds are dealt alternately to the GPUs. Each process checkpoints after every
# seed and stops cleanly if the next seed would not fit in the time left.
import os

parts = [SEEDS[i::N_GPU] for i in range(N_GPU)]
budget = SESSION_LIMIT_MIN - (time.time() - NOTEBOOK_T0) / 60 - 15
print("time budget for the chain: %.0f min" % budget)
stage_args = []
for c in CHAIN:
    stage_args += ["--stage", c + "=" + str(WHERE[c][2])]

procs = []
for g, part in enumerate(parts):
    if not part:
        continue
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g))
    log = open(OUT / ("chain_gpu%d.log" % g), "w")
    cmd = [sys.executable, "-u", "m4_chain.py", "--cover", str(COVER), *stage_args,
           "--epochs", *EPOCHS, "--seeds", *part, "--batch-size", BATCH,
           "--out", str(OUT / ("part%d" % g)), "--time-budget-min", "%.0f" % budget]
    print("GPU", g, "-> seeds", ", ".join(part))
    procs.append((g, subprocess.Popen(cmd, env=env, stdout=log, stderr=subprocess.STDOUT), log))

seen = {g: 0 for g, _, _ in procs}
while any(pr.poll() is None for _, pr, _ in procs):
    time.sleep(60)
    for g, _, _ in procs:
        lines = (OUT / ("chain_gpu%d.log" % g)).read_text(errors="replace").splitlines()
        for ln in lines[seen[g]:]:
            if "Warning" in ln or "warn" in ln or "autocast" in ln or "GradScaler" in ln:
                continue
            print("[gpu%d] %s" % (g, ln), flush=True)
        seen[g] = len(lines)
for g, pr, log in procs:
    log.close()
    print("GPU", g, "process exit code", pr.returncode)
print("elapsed: %.0f min" % ((time.time() - NOTEBOOK_T0) / 60))

In [ ]:
# ---- Cell 4: merge, summarise, package ---------------------------------
import shutil

import numpy as np

rows = []
(OUT / "scores").mkdir(parents=True, exist_ok=True)
merged = {}
for g in range(N_GPU):
    part = OUT / ("part%d" % g)
    if (part / "seed_results.json").exists():
        rows += json.loads((part / "seed_results.json").read_text())
    if (part / "scores").is_dir():
        for f in sorted((part / "scores").glob("*.csv")):
            lines = f.read_text().splitlines()
            merged.setdefault(f.name, [lines[0]])
            merged[f.name] += lines[1:]
for name, lines in merged.items():
    (OUT / "scores" / name).write_text(chr(10).join(lines) + chr(10))

# From-scratch reference on the same 5,000 images (earlier runs).
SCRATCH = {"SCRIPT_0.05": 0.2358, "SCRIPT_0.02": 0.3963, "SCRIPT_0.01": 0.4959}
DCTR = {"SCRIPT_0.05": 0.2304, "SCRIPT_0.02": 0.4031, "SCRIPT_0.01": 0.4723}

summary = []
print("stage".ljust(14) + "chain P_E".rjust(11) + "+/-".rjust(8) + "transfer".rjust(10)
      + "scratch".rjust(9) + "DCTR".rjust(8) + "  caught at 1% FA   best epochs")
print("-" * 92)
for c in CHAIN:
    rr = sorted((r for r in rows if r["stage"] == c), key=lambda r: r["seed"])
    if not rr:
        continue
    pe = [r["test_p_e"] for r in rr]
    tr = [r["transfer"]["test_p_e"] for r in rr if r["transfer"]]
    caught = [1 - r["md_at_fa1"] for r in rr]
    s = dict(stage=c, seeds=[r["seed"] for r in rr], p_e_mean=float(np.mean(pe)),
             p_e_std=float(np.std(pe)), p_e_per_seed=pe,
             transfer_p_e_mean=float(np.mean(tr)) if tr else None,
             caught_at_fa1_mean=float(np.mean(caught)),
             best_epoch_per_seed=[r["best_epoch"] for r in rr],
             val_p_e_per_seed=[r["val_p_e"] for r in rr],
             n_train_pairs=rr[0]["n_train_pairs"], n_val_pairs=rr[0]["n_val_pairs"],
             n_test_pairs=rr[0]["n_test_pairs"], epoch_selection="validation",
             from_scratch_p_e=SCRATCH.get(c), dctr_p_e=DCTR.get(c))
    summary.append(s)
    print(c.ljust(14) + ("%.4f" % s["p_e_mean"]).rjust(11) + ("%.4f" % s["p_e_std"]).rjust(8)
          + (("%.4f" % s["transfer_p_e_mean"]) if tr else "-").rjust(10)
          + (("%.4f" % SCRATCH[c]) if c in SCRATCH else "-").rjust(9)
          + (("%.4f" % DCTR[c]) if c in DCTR else "-").rjust(8)
          + ("%.3f" % s["caught_at_fa1_mean"]).rjust(12) + "        "
          + str(s["best_epoch_per_seed"]))
(OUT / "chain_results.json").write_text(json.dumps(summary, indent=2))
(OUT / "seed_results.json").write_text(json.dumps(rows, indent=2))

done = sorted({r["seed"] for r in rows if r["stage"] == CHAIN[-1]})
print()
print("seeds finished:", done, "of", SEEDS)
print("transfer = the network from the previous stage applied unchanged")
print("best epoch -1 = fine-tuning never beat the arriving weights on validation")

null = json.loads((OUT / "null" / "null_test.json").read_text())
notes = [
    "# Chain run notes", "",
    "- Chain: " + " -> ".join(CHAIN) + ", epochs " + "/".join(EPOCHS),
    "- Hardware: %d x %s; torch %s" % (N_GPU, torch.cuda.get_device_name(0), torch.__version__),
    "- EfficientNet-B0, ImageNet-pretrained, 1 channel, 512x512, no resize; batch %s, "
    "AdamW lr 1e-4 (fresh optimiser per stage), AMP" % BATCH,
    "- One paired 40/10/50 split per seed, shared by all stages; epoch chosen on "
    "validation, arriving weights included as a candidate",
    "- CNN null P_E: %.4f (chance %.4f)" % (null["p_e_mean"], null["null_expected"]),
    "- Seeds finished: %s of %s" % (done, SEEDS),
    "- Total minutes: %.0f" % ((time.time() - NOTEBOOK_T0) / 60),
]
(OUT / "NOTES.md").write_text(chr(10).join(notes) + chr(10))

stage = pathlib.Path("/kaggle/working/_zip")
if stage.exists():
    shutil.rmtree(stage)
stage.mkdir()
shutil.copytree(OUT / "scores", stage / "scores")
shutil.copytree(OUT / "null", stage / "null")
for f in list(OUT.glob("*.log")) + list(OUT.glob("*.json")) + [OUT / "NOTES.md"]:
    shutil.copy2(f, stage / f.name)
shutil.make_archive("/kaggle/working/chain_results", "zip", stage)
shutil.rmtree(stage)
print()
print("wrote /kaggle/working/chain_results.zip - download it from the Output tab")